# Integrált TK-101 esettanulmány

System model → FTA → sensor evidence → Bayesian update → dynamic risk → maintenance option.

In [ ]:
from pathlib import Path
import pandas as pd
from safetycourse.model_io import load_yaml,validate_system_model,generate_fault_tree
from safetycourse.fta import top_event_probability,minimal_cut_sets
from safetycourse.diagnosis import residual,ewma
from safetycourse.bayes import BinaryNode,BinaryBayesNet
from safetycourse.maintenance import compare_maintenance_options

In [ ]:
model=load_yaml('../data/pundit_case/system_model.yaml')
print('validation',validate_system_model(model))
tree=generate_fault_tree(model,'OVERFLOW')
print('baseline',top_event_probability(tree))
print('cuts',[sorted(c) for c in minimal_cut_sets(tree)])

In [ ]:
df=pd.read_csv('../data/pundit_case/sensor_data.csv')
df['residual']=residual(df.lt101_measured_pct,df.true_level_pct)
df['ewma']=ewma(df.residual,0.12)
df['mismatch']=df.ewma.abs()>4
net=BinaryBayesNet([BinaryNode('F',[],{():0.02}),BinaryNode('E',['F'],{(False,):0.05,(True,):0.90})])
df['p_fault']=[net.query('F',{'E':bool(v)})[True] for v in df.mismatch]

In [ ]:
p_ch=0.08; p_auto=1-(1-0.01)*(1-0.005); p_op=0.04
df['p_overflow']=df.p_fault.map(lambda pf:p_ch*p_auto*(1-(1-pf)*(1-p_op)))
current=float(df.p_overflow.iloc[-1])
print('current dynamic overflow probability',current)

In [ ]:
options=compare_maintenance_options([
 {'name':'maintain_now','maintenance_cost':250000,'production_loss':800000,'hazard_probability':0.001,'consequence_cost':50000000},
 {'name':'wait_for_window','maintenance_cost':250000,'production_loss':100000,'hazard_probability':max(0.02,current),'consequence_cost':50000000},
])
options

A `wait_for_window` hazard probability külön scenario-paraméter; nem automatikus skálázása a pillanatnyi kockázatnak. Valós alkalmazáshoz waiting-horizon modell kell.